# 04 — PESADO TPU/GPU: coloredor neuronal JAX (usa lo que no tenés)

SAT no corre en TPU/GPU (branching simbólico CPU). Esto SÍ: búsqueda neuronal masiva.
Valor: criba rápida de candidatos grandes (0 violaciones = k-coloreable = se DESCARTA
sin quemar horas de kissat; piso alto = va a SAT).
Pasos: Entorno de ejecución → Cambiar tipo → TPU v5e-1 (o T4 GPU) → Ejecutar todo.
Pegá los JSON impresos en el chat de Grafito.


In [ ]:
import subprocess, sys
try:
    import jax
    print('jax', jax.__version__, jax.devices())
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "jax[tpu]",
                           "-f", "https://storage.googleapis.com/jax-releases/libtpu_releases.html"])
    import jax
    print('jax', jax.__version__, jax.devices())


## Demo (triángulo: k2 piso ≥1, k3 → 0)


In [ ]:
import sys
#!/usr/bin/env python3
"""nn_jax.py — coloreo probabilístico Hadwiger-Nelson en JAX (CPU acá, TPU en Colab).

Modelo (igual idea que lab/colG_pack.py, pero vectorizado + pmap/vmap):
  logits (R, N, k) -> softmax -> p[r,i,c]. Loss[r] = media sobre aristas de
  sum_c p[r,i,c]*p[r,j,c]  -  entropy_coef * entropía_media (para no colapsar).
  Adam a mano (sin optax) + jit + vmap sobre restarts. En TPU vmap corre en los
  8 cores del host.

Valor honesto: BUSCA coloreos rápido (heurística, NO prueba). Si llega a
0 violaciones con argmax, el grafo es k-coloreable (testigo de descarte para
candidatos grandes donde kissat tarda horas). Si queda en piso >0, NO prueba
nada: el candidato va a kissat.

Formatos .edge: 'e a b' o 'a b', 1-based por defecto (--zero-based para 0-based).

Ejemplos:
  python nn_jax.py --demo
  python nn_jax.py --edges /tmp/opencode/874.edge --k 5 --restarts 8 --steps 3000
"""

import argparse
import json
import sys
import time

import jax
import jax.numpy as jnp


def load_edges(path, zero_based=False):
    edges = []
    seen = set()
    with open(path) as f:
        for line in f:
            p = line.split()
            if not p or p[0].startswith(("c", "p", "#")):
                continue
            if p[0] == "e" and len(p) >= 3:
                a, b = int(p[1]), int(p[2])
            elif len(p) >= 2:
                try:
                    a, b = int(p[0]), int(p[1])
                except ValueError:
                    continue
            else:
                continue
            if not zero_based:
                a, b = a - 1, b - 1
            if a == b:
                continue
            if a > b:
                a, b = b, a
            if (a, b) not in seen:
                seen.add((a, b))
                edges.append((a, b))
    return edges


def train(key, ei, ej, n, k, steps, lr, entropy_coef, temp_init):
    rkey, skey = jax.random.split(key)
    logits = jax.random.normal(rkey, (n, k)) * temp_init

    m = jnp.zeros_like(logits)
    v = jnp.zeros_like(logits)
    b1, b2, eps = 0.9, 0.999, 1e-8

    def loss_fn(lg, beta):
        p = jax.nn.softmax(lg, axis=-1)
        same = jnp.sum(p[ei] * p[ej], axis=-1).mean()
        ent = -(p * jnp.log(p + 1e-12)).sum(-1).mean()
        return same - beta * ent

    # schedule coseno = lab/colG_pack.py::entropy_beta (el que plantó 874 en 0 en torch)
    t_all = jnp.arange(steps)
    betas = entropy_coef * 0.5 * (1.0 + jnp.cos(jnp.pi * t_all / steps))

    @jax.jit
    def step(carry, tb):
        t, beta = tb
        lg, m, v = carry
        loss, g = jax.value_and_grad(loss_fn)(lg, beta)
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g * g
        mh = m / (1 - b1 ** (t + 1))
        vh = v / (1 - b2 ** (t + 1))
        lg = lg - lr * mh / (jnp.sqrt(vh) + eps)
        return (lg, m, v), loss

    (lg, _, _), losses = jax.lax.scan(step, (logits, m, v), (t_all, betas))
    p = jax.nn.softmax(lg, axis=-1)
    hard = jnp.argmax(p, axis=-1)
    viol = jnp.sum(hard[ei] == hard[ej])
    return losses[-1], viol, hard


def refine_numpy(hard, ei, ej, k, max_passes=50, kicks=0, kick_size=5, seed=0):
    """Descenso greedy discreto (numpy, CPU) + basin hopping opcional.
    Greedy solo remata (~40→~35); con kicks (perturbación aleatoria +
    greedy, quedándose con lo mejor) escapa de óptimos locales: 12→0."""
    import numpy as np

    col0 = np.array(hard, dtype=np.int64).tolist()
    ei_l = np.array(ei, dtype=np.int64).tolist()
    ej_l = np.array(ej, dtype=np.int64).tolist()
    n = max(len(col0), max(ei_l) + 1, max(ej_l) + 1)
    adj = [[] for _ in range(n)]
    for a, b in zip(ei_l, ej_l):
        adj[a].append(b)
        adj[b].append(a)

    def viol_of(col):
        return sum(1 for a, b in zip(ei_l, ej_l) if col[a] == col[b])

    def greedy(col):
        viol = viol_of(col)
        for _ in range(max_passes):
            improved = False
            for v in range(n):
                cur = col[v]
                cnt = [0] * k
                for u in adj[v]:
                    cnt[col[u]] += 1
                best_c = min(range(k), key=lambda c: cnt[c])
                if cnt[best_c] < cnt[cur]:
                    col[v] = best_c
                    viol += cnt[best_c] - cnt[cur]
                    improved = True
            if not improved:
                break
        return col, viol

    rng = np.random.default_rng(seed)
    best_col, best_v = greedy(list(col0))
    for _ in range(kicks):
        cand = list(best_col)
        for v in rng.integers(0, n, size=kick_size).tolist():
            cand[v] = int(rng.integers(0, k))
        cand, v = greedy(cand)
        if v < best_v:
            best_col, best_v = cand, v
            if best_v == 0:
                break
    return best_v, best_col


def run(
    edges, k, restarts, steps, lr, entropy_coef, temp_init, seed, kicks=0, kick_size=5
):
    nodes = set()
    for a, b in edges:
        nodes.add(a)
        nodes.add(b)
    n = max(nodes) + 1
    ei = jnp.array([a for a, _ in edges], dtype=jnp.int32)
    ej = jnp.array([b for _, b in edges], dtype=jnp.int32)
    keys = jax.random.split(jax.random.PRNGKey(seed), restarts)
    vtrain = jax.vmap(
        lambda key: train(key, ei, ej, n, k, steps, lr, entropy_coef, temp_init)
    )
    t0 = time.time()
    losses, viols, hards = vtrain(keys)
    dt = time.time() - t0
    losses = [float(x) for x in losses]
    viols = [int(x) for x in viols]
    best = int(jnp.argmin(jnp.array(viols)))
    t1 = time.time()
    refined, refined_col = refine_numpy(
        hards[best], ei, ej, k, kicks=kicks, kick_size=kick_size, seed=seed
    )
    refine_s = round(time.time() - t1, 3)
    return {
        "n": n,
        "edges": len(edges),
        "k": k,
        "restarts": restarts,
        "steps": steps,
        "time_s": round(dt, 3),
        "devices": [str(d) for d in jax.devices()],
        "best_restart": best,
        "best_violations": viols[best],
        "refined_violations": refined,
        "refined_coloring": refined_col,
        "refine_s": refine_s,
        "best_loss": losses[best],
        "all_violations": viols,
    }


def demo():
    # triángulo: k2 piso 1/3 (espejo UNSAT), k3 -> 0 (espejo SAT)
    tri = [(0, 1), (1, 2), (0, 2)]
    k2 = run(tri, 2, 4, 800, 0.05, 0.02, 1.0, 0)
    k3 = run(tri, 3, 4, 800, 0.05, 0.02, 1.0, 1)
    ok = k2["best_violations"] >= 1 and k3["best_violations"] == 0
    result = {"triangle_k2": k2, "triangle_k3": k3, "demo_ok": bool(ok)}
    print(json.dumps(result))
    return 0 if ok else 1


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--demo", action="store_true")
    ap.add_argument("--edges", default=None)
    ap.add_argument("--zero-based", action="store_true")
    ap.add_argument("--k", type=int, default=5)
    ap.add_argument("--restarts", type=int, default=8)
    ap.add_argument("--steps", type=int, default=3000)
    ap.add_argument("--lr", type=float, default=0.15)
    ap.add_argument("--entropy", type=float, default=0.05)
    ap.add_argument("--temp", type=float, default=1.0)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--kicks", type=int, default=0)
    ap.add_argument("--kick-size", type=int, default=5)
    ap.add_argument(
        "--x64",
        action="store_true",
        help="float64 (igual que el torch ganador; en TPU emula lento: usar T4)",
    )
    a = ap.parse_args()
    if a.x64:
        jax.config.update("jax_enable_x64", True)
    if a.demo:
        return demo()
    if not a.edges:
        ap.error("--edges requerido sin --demo")
    edges = load_edges(a.edges, a.zero_based)
    print(
        json.dumps(
            run(
                edges,
                a.k,
                a.restarts,
                a.steps,
                a.lr,
                a.entropy,
                a.temp,
                a.seed,
                kicks=a.kicks,
                kick_size=a.kick_size,
            )
        )
    )
    return 0



sys.argv=["nn_jax.py","--demo"]
_ = main()


## Escala x64 en T4 (el torch ganador usaba float64)
IMPORTANTE: cambiá el entorno a **T4 GPU** (la TPU emula float64 lento).
64 restarts × 20000 steps + 100 kicks. Esperado: `refined_violations` → 0.
Si da 0, el `refined_coloring` se verifica arista por arista en local.


In [ ]:
EDGE874 = "p edge 874 4461\ne 1 2\ne 1 3\ne 1 4\ne 1 5\ne 1 6\ne 1 7\ne 1 9\ne 1 14\ne 1 16\ne 1 24\ne 1 33\ne 1 41\ne 1 43\ne 1 48\ne 1 51\ne 1 57\ne 1 59\ne 1 68\ne 1 77\ne 1 85\ne 1 86\ne 1 92\ne 1 94\ne 1 101\ne 1 114\ne 1 120\ne 1 125\ne 1 129\ne 1 135\ne 1 137\ne 2 6\ne 2 7\ne 2 25\ne 2 26\ne 2 27\ne 2 28\ne 2 29\ne 2 30\ne 2 31\ne 2 32\ne 2 140\ne 2 141\ne 2 146\ne 2 147\ne 2 150\ne 2 157\ne 2 158\ne 2 159\ne 2 160\ne 2 164\ne 2 170\ne 2 177\ne 2 198\ne 2 428\ne 2 432\ne 2 433\ne 2 452\ne 2 453\ne 2 454\ne 2 455\ne 2 456\ne 2 457\ne 2 458\ne 2 567\ne 2 568\ne 2 575\ne 2 576\ne 2 580\ne 2 581\ne 2 588\ne 2 589\ne 2 591\ne 2 593\ne 2 597\ne 2 603\ne 2 608\ne 2 613\ne 2 616\ne 2 637\ne 2 638\ne 3 5\ne 3 7\ne 3 10\ne 3 17\ne 3 108\ne 3 109\ne 3 110\ne 3 111\ne 3 112\ne 3 113\ne 3 213\ne 3 218\ne 3 223\ne 3 227\ne 3 269\ne 3 274\ne 3 281\ne 3 287\ne 3 290\ne 3 297\ne 3 302\ne 3 308\ne 3 309\ne 4 5\ne 4 6\ne 4 40\ne 4 47\ne 4 55\ne 4 64\ne 4 80\ne 4 87\ne 4 96\ne 4 102\ne 4 347\ne 4 349\ne 4 351\ne 4 353\ne 4 362\ne 4 369\ne 4 371\ne 4 379\ne 4 393\ne 4 398\ne 5 12\ne 5 20\ne 5 36\ne 5 44\ne 5 60\ne 5 133\ne 5 134\ne 5 295\ne 5 298\ne 5 302\ne 5 318\ne 5 334\ne 5 339\ne 5 345\ne 5 350\ne 5 353\ne 5 355\ne 5 357\ne 5 431\ne 6 84\ne 6 90\ne 6 98\ne 6 105\ne 6 118\ne 6 122\ne 6 126\ne 6 130\ne 6 162\ne 6 167\ne 6 367\ne 6 368\ne 6 369\ne 6 406\ne 6 408\ne 6 413\ne 6 417\ne 6 426\ne 6 427\ne 7 69\ne 7 70\ne 7 71\ne 7 72\ne 7 73\ne 7 74\ne 7 75\ne 7 76\ne 7 142\ne 7 147\ne 7 188\ne 7 189\ne 7 202\ne 7 206\ne 7 210\ne 7 217\ne 7 220\ne 7 222\ne 7 227\ne 7 232\ne 7 237\ne 7 238\ne 7 239\ne 7 241\ne 8 11\ne 8 33\ne 8 57\ne 8 82\ne 8 145\ne 8 179\ne 8 259\ne 8 314\ne 8 358\ne 9 50\ne 9 51\ne 9 52\ne 9 53\ne 9 54\ne 9 55\ne 9 56\ne 9 57\ne 9 140\ne 9 162\ne 9 174\ne 9 203\ne 9 209\ne 10 12\ne 10 32\ne 10 57\ne 10 95\ne 10 110\ne 10 160\ne 10 229\ne 10 231\ne 10 261\ne 10 262\ne 10 263\ne 11 13\ne 11 50\ne 11 57\ne 11 77\ne 11 116\ne 11 153\ne 11 155\ne 11 200\ne 11 261\ne 11 315\ne 11 360\ne 11 421\ne 12 57\ne 12 71\ne 12 76\ne 12 95\ne 12 98\ne 12 314\ne 12 315\ne 12 316\ne 12 317\ne 12 318\ne 12 368\ne 12 379\ne 13 53\ne 13 57\ne 13 114\ne 13 197\ne 13 201\ne 13 263\ne 13 317\ne 14 42\ne 14 57\ne 14 67\ne 14 75\ne 14 82\ne 14 110\ne 14 116\ne 14 125\ne 14 318\ne 14 362\ne 14 399\ne 15 19\ne 15 43\ne 15 68\ne 15 89\ne 15 154\ne 15 184\ne 15 234\ne 15 260\ne 15 264\ne 15 301\ne 15 319\ne 15 363\ne 16 58\ne 16 59\ne 16 60\ne 16 61\ne 16 62\ne 16 63\ne 16 64\ne 16 65\ne 16 66\ne 16 67\ne 16 68\ne 16 141\ne 16 167\ne 16 202\ne 16 223\ne 17 20\ne 17 39\ne 17 55\ne 17 68\ne 17 79\ne 17 96\ne 17 111\ne 17 142\ne 17 203\ne 17 223\ne 17 264\ne 17 265\ne 17 266\ne 17 267\ne 17 268\ne 18 21\ne 18 40\ne 18 68\ne 18 77\ne 18 102\ne 18 117\ne 18 143\ne 18 152\ne 18 155\ne 18 266\ne 18 300\ne 18 301\ne 18 320\ne 18 364\ne 18 402\ne 19 22\ne 19 58\ne 19 68\ne 19 86\ne 19 121\ne 19 144\ne 19 204\ne 19 234\ne 19 235\ne 19 267\ne 19 300\ne 19 322\ne 19 326\ne 19 365\ne 19 403\ne 20 60\ne 20 68\ne 20 319\ne 20 320\ne 20 321\ne 20 370\ne 20 377\ne 21 61\ne 21 68\ne 21 79\ne 21 84\ne 21 102\ne 21 106\ne 21 114\ne 21 145\ne 21 184\ne 21 186\ne 21 195\ne 21 314\ne 21 321\ne 21 326\ne 21 333\ne 21 366\ne 21 368\ne 21 371\ne 21 391\ne 21 404\ne 21 418\ne 22 62\ne 22 68\ne 22 120\ne 22 205\ne 22 268\ne 22 322\ne 22 405\ne 23 63\ne 23 68\ne 23 96\ne 23 98\ne 23 125\ne 23 331\ne 23 369\ne 24 49\ne 24 68\ne 24 76\ne 24 83\ne 24 89\ne 24 111\ne 24 117\ne 24 121\ne 24 129\ne 24 146\ne 24 406\ne 25 29\ne 25 47\ne 25 65\ne 25 77\ne 25 87\ne 25 103\ne 25 118\ne 25 143\ne 25 148\ne 25 150\ne 25 190\ne 25 191\ne 25 207\ne 25 232\ne 25 276\ne 26 30\ne 26 86\ne 26 108\ne 26 122\ne 26 144\ne 26 151\ne 26 158\ne 26 163\ne 26 192\ne 26 193\ne 26 208\ne 26 218\ne 26 228\ne 26 238\ne 26 276\ne 26 296\ne 27 31\ne 27 55\ne 27 67\ne 27 94\ne 27 102\ne 27 110\ne 27 126\ne 27 152\ne 27 160\ne 27 194\ne 27 195\ne 27 209\ne 27 223\ne 27 241\ne 27 285\ne 28 32\ne 28 101\ne 28 130\ne 28 153\ne 28 164\ne 28 196\ne 28 197\ne 28 224\ne 28 229\ne 29 69\ne 29 114\ne 29 145\ne 29 165\ne 29 171\ne 29 179\ne 29 425\ne 30 70\ne 30 87\ne 30 91\ne 30 108\ne 30 113\ne 30 120\ne 30 149\ne 30 154\ne 30 161\ne 30 166\ne 30 180\ne 30 184\ne 30 198\ne 30 367\ne 30 371\ne 30 397\ne 31 71\ne 31 102\ne 31 125\ne 31 155\ne 31 379\ne 32 72\ne 32 103\ne 32 106\ne 32 110\ne 32 129\ne 32 146\ne 32 156\ne 32 172\ne 32 199\ne 32 368\ne 33 77\ne 33 78\ne 33 79\ne 33 80\ne 33 81\ne 33 82\ne 33 83\ne 33 84\ne 33 85\ne 33 150\ne 33 210\ne 33 269\ne 33 283\ne 34 37\ne 34 74\ne 34 85\ne 34 86\ne 34 109\ne 34 123\ne 34 151\ne 34 227\ne 34 251\ne 34 311\ne 35 39\ne 35 65\ne 35 76\ne 35 85\ne 35 101\ne 35 108\ne 35 117\ne 35 131\ne 35 153\ne 35 163\ne 35 166\ne 35 181\ne 35 194\ne 35 212\ne 35 223\ne 35 232\ne 35 236\ne 35 270\ne 35 283\ne 35 299\ne 35 306\ne 35 315\ne 35 327\ne 35 336\ne 36 40\ne 36 85\ne 36 133\ne 36 213\ne 36 233\ne 36 235\ne 36 266\ne 36 306\ne 36 322\ne 36 323\ne 36 324\ne 37 78\ne 37 85\ne 37 120\ne 37 154\ne 37 214\ne 37 311\ne 37 319\ne 37 336\ne 37 407\ne 38 42\ne 38 85\ne 38 125\ne 38 138\ne 38 155\ne 38 215\ne 38 271\ne 38 315\ne 38 320\ne 38 383\ne 38 389\ne 39 79\ne 39 85\ne 39 108\ne 39 129\ne 39 144\ne 39 156\ne 39 216\ne 39 272\ne 39 322\ne 39 370\ne 39 378\ne 39 389\ne 40 80\ne 40 85\ne 40 109\ne 40 113\ne 40 117\ne 40 175\ne 40 323\ne 40 370\ne 40 371\ne 41 85\ne 41 91\ne 41 99\ne 41 106\ne 41 112\ne 41 123\ne 41 127\ne 41 131\ne 41 133\ne 41 135\ne 41 138\ne 41 157\ne 41 173\ne 41 188\ne 41 371\ne 41 408\ne 42 82\ne 42 85\ne 42 194\ne 42 273\ne 42 324\ne 42 378\ne 42 383\ne 42 409\ne 43 86\ne 43 87\ne 43 88\ne 43 89\ne 43 90\ne 43 91\ne 43 92\ne 43 158\ne 43 207\ne 43 217\ne 43 251\ne 43 274\ne 43 288\ne 43 295\ne 44 47\ne 44 74\ne 44 84\ne 44 92\ne 44 115\ne 44 122\ne 44 134\ne 44 218\ne 44 232\ne 44 235\ne 44 252\ne 44 295\ne 44 304\ne 44 325\ne 44 326\ne 44 327\ne 45 92\ne 45 114\ne 45 136\ne 45 186\ne 45 191\ne 45 196\ne 45 276\ne 45 325\ne 45 372\ne 46 49\ne 46 92\ne 46 129\ne 46 139\ne 46 144\ne 46 145\ne 46 245\ne 46 246\ne 46 277\ne 46 326\ne 47 87\ne 47 92\ne 47 115\ne 47 175\ne 47 198\ne 47 372\ne 48 92\ne 48 107\ne 48 113\ne 48 119\ne 48 132\ne 48 134\ne 48 136\ne 48 137\ne 48 139\ne 48 159\ne 48 189\ne 49 89\ne 49 92\ne 49 153\ne 49 196\ne 49 219\ne 49 226\ne 49 278\ne 49 327\ne 49 414\ne 50 53\ne 50 77\ne 50 152\ne 50 153\ne 50 253\ne 50 273\ne 50 373\ne 50 409\ne 51 93\ne 51 94\ne 51 95\ne 51 96\ne 51 97\ne 51 98\ne 51 99\ne 51 160\ne 51 203\ne 51 220\ne 51 282\ne 51 289\ne 51 679\ne 52 54\ne 52 75\ne 52 101\ne 52 110\ne 52 172\ne 52 209\ne 52 227\ne 52 289\ne 53 56\ne 53 93\ne 53 114\ne 53 195\ne 53 197\ne 53 221\ne 53 234\ne 53 279\ne 53 328\ne 53 375\ne 53 411\ne 54 95\ne 54 129\ne 54 250\ne 54 253\ne 54 255\ne 55 96\ne 55 110\ne 55 146\ne 55 162\ne 55 373\ne 55 374\ne 55 375\ne 55 376\ne 55 377\ne 55 378\ne 55 379\ne 56 97\ne 56 135\ne 56 216\ne 56 280\ne 56 378\ne 57 174\ne 57 379\ne 57 413\ne 58 62\ne 58 86\ne 58 163\ne 58 235\ne 58 257\ne 58 278\ne 58 306\ne 58 329\ne 58 380\ne 58 414\ne 59 100\ne 59 101\ne 59 102\ne 59 103\ne 59 104\ne 59 105\ne 59 106\ne 59 107\ne 59 164\ne 59 209\ne 59 222\ne 59 281\ne 59 298\ne 60 64\ne 60 83\ne 60 98\ne 60 117\ne 60 126\ne 60 223\ne 60 282\ne 60 298\ne 60 329\ne 60 330\ne 60 331\ne 61 65\ne 61 84\ne 61 114\ne 61 130\ne 61 165\ne 61 224\ne 61 283\ne 61 330\ne 61 381\ne 62 66\ne 62 100\ne 62 120\ne 62 166\ne 62 225\ne 62 284\ne 62 306\ne 62 307\ne 62 382\ne 62 384\ne 62 387\ne 62 415\ne 63 67\ne 63 125\ne 63 285\ne 63 358\ne 63 360\ne 63 392\ne 64 102\ne 64 167\ne 64 331\ne 64 380\ne 64 381\ne 64 382\ne 64 383\ne 64 418\ne 65 103\ne 65 117\ne 65 130\ne 65 135\ne 65 143\ne 65 146\ne 65 153\ne 65 168\ne 65 215\ne 65 257\ne 65 258\ne 65 360\ne 65 373\ne 65 383\ne 65 387\ne 65 416\ne 65 425\ne 66 104\ne 66 137\ne 66 169\ne 66 226\ne 66 286\ne 66 384\ne 67 126\ne 67 392\ne 68 142\ne 68 170\ne 68 417\ne 69 73\ne 69 90\ne 69 106\ne 69 114\ne 69 122\ne 69 131\ne 69 171\ne 69 201\ne 69 221\ne 69 224\ne 69 228\ne 69 232\ne 69 240\ne 69 288\ne 70 74\ne 70 91\ne 70 120\ne 70 133\ne 70 205\ne 70 214\ne 70 225\ne 70 233\ne 70 238\ne 70 244\ne 70 295\ne 70 303\ne 70 308\ne 70 514\ne 71 75\ne 71 98\ne 71 125\ne 71 130\ne 71 215\ne 71 229\ne 71 241\ne 71 259\ne 71 289\ne 71 298\ne 71 309\ne 72 76\ne 72 129\ne 72 172\ne 72 216\ne 72 234\ne 72 260\ne 72 299\ne 73 108\ne 73 135\ne 73 149\ne 73 153\ne 73 156\ne 73 188\ne 73 230\ne 73 242\ne 73 245\ne 73 249\ne 73 253\ne 74 109\ne 74 122\ne 74 133\ne 74 137\ne 74 151\ne 74 173\ne 74 189\ne 74 198\ne 74 226\ne 74 235\ne 74 243\ne 74 246\ne 74 254\ne 74 257\ne 74 407\ne 75 110\ne 75 130\ne 75 162\ne 76 111\ne 76 131\ne 76 142\ne 76 146\ne 76 174\ne 76 219\ne 76 231\ne 76 236\ne 76 250\ne 77 114\ne 77 115\ne 77 116\ne 77 117\ne 77 118\ne 77 119\ne 77 232\ne 77 290\ne 77 341\ne 77 347\ne 78 81\ne 78 113\ne 78 120\ne 78 134\ne 78 233\ne 78 302\ne 78 323\ne 78 344\ne 79 83\ne 79 106\ne 79 129\ne 79 133\ne 79 234\ne 79 244\ne 79 246\ne 79 255\ne 79 259\ne 79 266\ne 79 283\ne 79 291\ne 79 298\ne 79 321\ne 79 332\ne 79 341\ne 79 352\ne 79 363\ne 79 365\ne 79 375\ne 79 388\ne 80 84\ne 80 304\ne 80 307\ne 80 321\ne 80 330\ne 80 347\ne 80 358\ne 80 384\ne 80 385\ne 81 115\ne 81 137\ne 81 175\ne 81 235\ne 81 292\ne 81 323\ne 81 365\ne 81 380\ne 82 116\ne 82 145\ne 82 176\ne 82 293\ne 82 333\ne 82 375\ne 82 381\ne 83 117\ne 83 133\ne 83 225\ne 83 226\ne 83 236\ne 83 294\ne 83 384\ne 83 418\ne 84 118\ne 84 134\ne 84 150\ne 84 252\ne 84 385\ne 84 418\ne 85 177\ne 85 213\ne 85 237\ne 85 251\ne 86 120\ne 86 121\ne 86 122\ne 86 123\ne 86 238\ne 86 288\ne 86 334\ne 86 349\ne 86 498\ne 87 90\ne 87 113\ne 87 136\ne 87 276\ne 87 295\ne 87 307\ne 87 349\ne 87 356\ne 87 359\ne 87 363\ne 87 386\ne 87 387\ne 88 91\ne 88 135\ne 88 171\ne 88 258\ne 88 260\ne 88 296\ne 88 419\ne 89 121\ne 89 153\ne 89 154\ne 89 178\ne 89 225\ne 89 310\ne 89 336\ne 89 387\ne 90 122\ne 90 136\ne 90 148\ne 90 149\ne 90 158\ne 90 189\ne 90 252\ne 90 419\ne 91 123\ne 91 147\ne 91 149\ne 91 173\ne 91 208\ne 91 249\ne 91 251\ne 92 173\ne 92 207\ne 92 218\ne 92 239\ne 93 97\ne 93 114\ne 93 179\ne 93 234\ne 93 240\ne 93 312\ne 93 314\ne 93 388\ne 93 420\ne 94 124\ne 94 125\ne 94 126\ne 94 127\ne 94 241\ne 94 282\ne 94 340\ne 95 129\ne 95 250\ne 95 289\ne 95 302\ne 95 316\ne 96 98\ne 96 285\ne 96 309\ne 96 316\ne 96 388\ne 96 389\ne 96 390\ne 97 99\ne 97 124\ne 97 135\ne 97 181\ne 97 215\ne 97 216\ne 97 242\ne 97 261\ne 97 306\ne 97 337\ne 97 389\ne 97 421\ne 98 126\ne 98 142\ne 98 160\ne 98 162\ne 98 420\ne 98 421\ne 99 127\ne 99 183\ne 99 294\ne 99 338\ne 99 390\ne 100 104\ne 100 120\ne 100 184\ne 100 244\ne 100 307\ne 100 313\ne 100 319\ne 100 321\ne 100 422\ne 101 128\ne 101 129\ne 101 130\ne 101 131\ne 101 132\ne 101 283\ne 101 289\ne 101 297\ne 101 339\ne 101 351\ne 102 105\ne 102 285\ne 102 298\ne 102 340\ne 102 351\ne 102 391\ne 102 392\ne 103 106\ne 103 135\ne 103 185\ne 103 245\ne 103 261\ne 103 299\ne 103 305\ne 103 341\ne 103 391\ne 104 107\ne 104 128\ne 104 137\ne 104 186\ne 104 246\ne 104 300\ne 104 321\ne 104 325\ne 104 342\ne 104 402\ne 104 403\ne 104 423\ne 105 130\ne 105 143\ne 105 145\ne 105 164\ne 105 167\ne 105 392\ne 105 422\ne 105 423\ne 106 131\ne 106 142\ne 106 145\ne 106 150\ne 106 156\ne 106 187\ne 106 224\ne 106 234\ne 106 247\ne 106 277\ne 106 293\ne 106 313\ne 106 403\ne 106 411\ne 106 420\ne 107 132\ne 107 248\ne 107 301\ne 107 343\ne 107 402\ne 108 112\ne 108 135\ne 108 249\ne 108 280\ne 108 296\ne 108 299\ne 108 303\ne 108 355\ne 108 397\ne 109 113\ne 109 137\ne 109 286\ne 109 292\ne 109 300\ne 109 304\ne 109 308\ne 109 344\ne 109 349\ne 109 354\ne 109 357\ne 109 397\ne 110 273\ne 110 293\ne 110 305\ne 110 309\ne 110 318\ne 110 351\ne 111 250\ne 111 278\ne 111 294\ne 111 306\ne 111 316\ne 111 352\ne 112 133\ne 112 151\ne 112 154\ne 112 188\ne 112 213\ne 112 234\ne 112 236\ne 112 310\ne 112 311\ne 112 312\ne 113 134\ne 113 150\ne 113 154\ne 113 175\ne 113 189\ne 113 218\ne 113 233\ne 113 251\ne 113 301\ne 113 307\ne 113 313\ne 114 135\ne 114 136\ne 114 345\ne 114 393\ne 114 396\ne 115 119\ne 115 137\ne 115 304\ne 115 353\ne 115 385\ne 115 396\ne 116 190\ne 116 305\ne 117 271\ne 117 273\ne 117 306\ne 117 330\ne 117 341\ne 117 346\ne 117 351\ne 117 355\ne 117 370\ne 117 383\ne 117 394\ne 117 414\ne 117 415\ne 117 421\ne 118 191\ne 118 325\ne 118 347\ne 118 356\ne 118 383\ne 118 391\ne 118 402\ne 118 409\ne 119 136\ne 119 252\ne 119 307\ne 119 348\ne 119 385\ne 119 395\ne 119 415\ne 119 422\ne 120 137\ne 120 308\ne 120 367\ne 121 192\ne 121 310\ne 122 325\ne 122 349\ne 122 355\ne 122 367\ne 122 396\ne 122 401\ne 122 403\ne 122 410\ne 122 414\ne 123 193\ne 123 249\ne 123 277\ne 123 278\ne 123 397\ne 124 127\ne 124 135\ne 124 194\ne 124 253\ne 124 305\ne 124 306\ne 124 324\ne 124 373\ne 125 138\ne 125 309\ne 125 340\ne 125 399\ne 125 553\ne 126 318\ne 126 377\ne 126 552\ne 127 138\ne 127 195\ne 127 255\ne 127 293\ne 127 294\ne 127 321\ne 127 328\ne 128 132\ne 128 137\ne 128 196\ne 128 257\ne 128 325\ne 128 327\ne 128 380\ne 128 383\ne 128 424\ne 129 139\ne 129 341\ne 129 350\ne 129 368\ne 129 398\ne 130 351\ne 130 368\ne 130 399\ne 130 424\ne 130 425\ne 131 197\ne 131 310\ne 131 317\ne 131 328\ne 131 352\ne 131 425\ne 132 139\ne 132 165\ne 132 166\ne 132 258\ne 132 383\ne 133 311\ne 133 338\ne 133 352\ne 133 354\ne 133 355\ne 133 371\ne 133 407\ne 134 343\ne 134 348\ne 134 356\ne 134 357\ne 134 367\ne 134 407\ne 135 355\ne 135 426\ne 136 191\ne 136 356\ne 136 369\ne 136 396\ne 136 400\ne 137 198\ne 137 357\ne 137 396\ne 137 427\ne 137 807\ne 138 259\ne 138 312\ne 138 317\ne 138 321\ne 138 358\ne 138 420\ne 139 145\ne 139 199\ne 139 260\ne 139 313\ne 139 363\ne 139 422\ne 140 160\ne 140 161\ne 140 162\ne 141 163\ne 141 164\ne 141 165\ne 141 166\ne 141 167\ne 141 168\ne 141 169\ne 141 170\ne 141 202\ne 142 156\ne 142 162\ne 142 170\ne 142 202\ne 142 203\ne 142 204\ne 142 205\ne 143 145\ne 143 170\ne 143 246\ne 143 248\ne 143 363\ne 143 402\ne 144 163\ne 144 170\ne 144 192\ne 144 204\ne 144 246\ne 144 247\ne 144 403\ne 145 165\ne 145 170\ne 145 187\ne 145 404\ne 145 408\ne 145 425\ne 146 170\ne 146 174\ne 146 178\ne 146 192\ne 146 379\ne 146 406\ne 146 425\ne 147 171\ne 147 172\ne 147 173\ne 147 174\ne 147 206\ne 147 207\ne 147 208\ne 147 209\ne 148 168\ne 148 185\ne 148 191\ne 148 207\ne 149 189\ne 149 408\ne 149 419\ne 150 175\ne 150 176\ne 150 177\ne 150 210\ne 150 218\ne 150 224\ne 151 154\ne 151 173\ne 151 177\ne 151 193\ne 151 206\ne 151 218\ne 152 155\ne 152 177\ne 152 195\ne 152 211\ne 152 273\ne 153 156\ne 153 168\ne 153 174\ne 153 177\ne 153 197\ne 153 202\ne 153 207\ne 153 212\ne 153 224\ne 153 245\ne 154 177\ne 154 214\ne 154 407\ne 155 177\ne 155 215\ne 155 416\ne 155 421\ne 156 177\ne 156 216\ne 156 273\ne 156 421\ne 157 177\ne 157 183\ne 157 187\ne 157 188\ne 157 193\ne 157 195\ne 157 197\ne 157 408\ne 158 178\ne 158 217\ne 158 228\ne 159 189\ne 159 198\ne 159 199\ne 160 179\ne 160 180\ne 160 181\ne 160 182\ne 160 183\ne 160 220\ne 160 223\ne 160 229\ne 161 180\ne 161 280\ne 161 412\ne 162 409\ne 162 410\ne 162 411\ne 162 412\ne 162 413\ne 163 166\ne 163 219\ne 163 414\ne 164 184\ne 164 185\ne 164 186\ne 164 187\ne 164 222\ne 165 168\ne 165 224\ne 166 169\ne 166 184\ne 166 225\ne 166 252\ne 166 327\ne 166 415\ne 166 418\ne 167 414\ne 167 415\ne 167 416\ne 167 417\ne 168 185\ne 168 409\ne 168 416\ne 169 186\ne 169 198\ne 169 226\ne 169 418\ne 170 417\ne 171 187\ne 171 197\ne 171 207\ne 171 228\ne 172 174\ne 172 245\ne 172 247\ne 173 198\ne 174 197\ne 174 231\ne 175 198\ne 175 235\ne 175 403\ne 175 414\ne 176 190\ne 176 411\ne 177 237\ne 178 192\ne 178 258\ne 178 359\ne 179 181\ne 179 240\ne 179 247\ne 179 259\ne 179 263\ne 179 283\ne 179 420\ne 180 182\ne 180 284\ne 180 390\ne 181 183\ne 181 194\ne 181 200\ne 181 242\ne 181 421\ne 182 198\ne 182 243\ne 182 286\ne 182 390\ne 183 195\ne 183 236\ne 184 186\ne 184 244\ne 184 252\ne 184 260\ne 184 299\ne 184 301\ne 184 326\ne 184 422\ne 185 187\ne 185 200\ne 185 245\ne 186 196\ne 186 198\ne 186 246\ne 186 423\ne 187 197\ne 187 247\ne 187 260\ne 188 237\ne 188 247\ne 188 255\ne 188 259\ne 189 239\ne 189 248\ne 189 251\ne 189 252\ne 189 256\ne 189 258\ne 189 260\ne 190 386\ne 191 416\ne 192 257\ne 192 278\ne 192 361\ne 192 374\ne 193 219\ne 193 230\ne 194 195\ne 194 253\ne 194 315\ne 194 378\ne 194 391\ne 194 409\ne 195 236\ne 195 255\ne 195 314\ne 196 198\ne 196 257\ne 196 414\ne 196 416\ne 196 424\ne 197 425\ne 198 371\ne 198 427\ne 199 260\ne 199 419\ne 200 201\ne 200 232\ne 200 261\ne 201 221\ne 201 263\ne 201 401\ne 202 222\ne 202 223\ne 202 224\ne 202 225\ne 202 226\ne 203 216\ne 203 227\ne 203 234\ne 203 250\ne 203 265\ne 204 205\ne 204 238\ne 204 267\ne 204 272\ne 204 277\ne 204 410\ne 205 225\ne 205 268\ne 205 272\ne 205 313\ne 205 319\ne 205 412\ne 206 227\ne 206 228\ne 206 229\ne 206 230\ne 206 231\ne 207 232\ne 207 245\ne 208 238\ne 208 249\ne 208 656\ne 209 241\ne 210 232\ne 210 233\ne 210 234\ne 210 235\ne 210 236\ne 210 237\ne 210 269\ne 211 215\ne 211 237\ne 211 241\ne 211 255\ne 211 266\ne 212 216\ne 212 231\ne 212 237\ne 212 249\ne 212 261\ne 212 270\ne 212 278\ne 213 237\ne 213 269\ne 213 270\ne 213 271\ne 213 272\ne 213 273\ne 214 233\ne 214 237\ne 214 264\ne 215 237\ne 215 259\ne 215 261\ne 215 266\ne 215 271\ne 215 373\ne 216 234\ne 216 237\ne 216 249\ne 216 272\ne 217 238\ne 217 239\ne 217 274\ne 217 648\ne 218 239\ne 218 274\ne 218 275\ne 218 276\ne 218 277\ne 218 278\ne 219 239\ne 219 278\ne 220 240\ne 220 241\ne 220 242\ne 220 243\ne 221 240\ne 221 279\ne 221 410\ne 222 244\ne 222 245\ne 222 246\ne 222 247\ne 222 248\ne 222 281\ne 223 236\ne 223 281\ne 223 282\ne 223 283\ne 223 284\ne 223 285\ne 223 286\ne 224 275\ne 224 283\ne 224 414\ne 225 226\ne 225 244\ne 225 270\ne 225 284\ne 226 246\ne 226 286\ne 226 414\ne 227 249\ne 227 250\ne 227 251\ne 227 287\ne 227 288\ne 227 289\ne 227 667\ne 228 230\ne 228 247\ne 228 288\ne 228 296\ne 229 262\ne 229 281\ne 229 289\ne 230 249\ne 231 250\ne 232 252\ne 232 290\ne 232 295\ne 232 299\ne 233 235\ne 233 251\ne 233 287\ne 233 295\ne 234 236\ne 234 247\ne 234 281\ne 234 288\ne 234 291\ne 234 299\ne 234 310\ne 235 292\ne 236 294\ne 236 314\ne 238 303\ne 240 242\ne 240 262\ne 241 253\ne 241 254\ne 241 255\ne 241 256\ne 241 298\ne 241 479\ne 242 253\ne 242 262\ne 242 270\ne 243 254\ne 243 338\ne 244 246\ne 244 264\ne 245 247\ne 245 299\ne 246 248\ne 246 257\ne 246 276\ne 246 300\ne 246 363\ne 248 258\ne 248 301\ne 249 288\ne 249 303\ne 250 270\ne 250 310\ne 252 307\ne 253 255\ne 253 270\ne 253 273\ne 253 341\ne 254 256\ne 254 342\ne 255 259\ne 255 279\ne 256 343\ne 257 258\ne 257 276\ne 257 278\ne 257 352\ne 257 387\ne 258 260\ne 259 263\ne 259 312\ne 259 375\ne 259 425\ne 260 313\ne 261 263\ne 261 290\ne 261 305\ne 261 315\ne 262 297\ne 263 279\ne 263 317\ne 263 359\ne 263 361\ne 264 267\ne 264 274\ne 264 319\ne 265 272\ne 265 287\ne 265 291\ne 266 290\ne 266 298\ne 266 306\ne 266 320\ne 266 373\ne 267 268\ne 267 374\ne 268 284\ne 268 308\ne 268 376\ne 269 290\ne 269 291\ne 269 292\ne 269 293\ne 269 294\ne 270 272\ne 270 297\ne 270 303\ne 270 306\ne 270 310\ne 271 273\ne 271 309\ne 271 312\ne 271 337\ne 272 291\ne 272 303\ne 272 322\ne 272 337\ne 273 293\ne 273 324\ne 273 421\ne 274 295\ne 274 296\ne 275 277\ne 275 297\ne 276 325\ne 277 278\ne 277 313\ne 277 326\ne 278 327\ne 279 280\ne 279 328\ne 279 374\ne 281 297\ne 281 298\ne 281 299\ne 281 300\ne 281 301\ne 282 294\ne 282 302\ne 282 306\ne 283 330\ne 283 380\ne 284 286\ne 284 308\ne 284 336\ne 285 309\ne 285 314\ne 285 315\ne 285 724\ne 286 300\ne 286 327\ne 286 380\ne 287 302\ne 287 303\ne 288 310\ne 289 309\ne 290 304\ne 290 305\ne 290 306\ne 290 307\ne 291 294\ne 291 311\ne 291 319\ne 291 328\ne 291 332\ne 292 304\ne 292 329\ne 293 305\ne 293 328\ne 293 330\ne 293 333\ne 293 420\ne 294 306\ne 294 311\ne 295 319\ne 295 334\ne 295 335\ne 295 336\ne 296 735\ne 297 310\ne 297 339\ne 298 339\ne 298 340\ne 298 341\ne 298 342\ne 298 343\ne 299 326\ne 299 335\ne 299 341\ne 300 301\ne 300 326\ne 300 332\ne 300 342\ne 301 343\ne 302 311\ne 302 344\ne 304 307\ne 304 349\ne 305 373\ne 306 322\ne 306 339\ne 306 346\ne 306 352\ne 307 348\ne 308 354\ne 309 312\ne 309 316\ne 309 351\ne 310 352\ne 311 354\ne 312 314\ne 312 332\ne 312 390\ne 313 319\ne 313 370\ne 313 403\ne 314 315\ne 314 333\ne 314 358\ne 314 390\ne 315 317\ne 315 360\ne 315 389\ne 315 391\ne 316 398\ne 317 328\ne 317 345\ne 317 420\ne 318 324\ne 318 333\ne 318 362\ne 318 368\ne 318 377\ne 319 336\ne 319 363\ne 320 321\ne 320 323\ne 320 340\ne 320 346\ne 320 364\ne 321 330\ne 321 332\ne 321 340\ne 321 345\ne 321 366\ne 321 370\ne 321 384\ne 321 398\ne 322 332\ne 322 350\ne 322 370\ne 323 344\ne 323 346\ne 323 353\ne 324 333\ne 324 373\ne 324 394\ne 325 345\ne 325 356\ne 325 372\ne 326 327\ne 326 335\ne 326 350\ne 326 397\ne 327 336\ne 327 380\ne 327 397\ne 327 418\ne 328 345\ne 328 375\ne 329 334\ne 329 380\ne 330 345\ne 330 351\ne 330 381\ne 330 420\ne 331 340\ne 331 353\ne 331 384\ne 331 394\ne 332 350\ne 332 354\ne 334 349\ne 335 336\ne 335 350\ne 336 387\ne 337 338\ne 337 355\ne 337 389\ne 338 390\ne 339 350\ne 339 351\ne 339 352\ne 340 353\ne 341 355\ne 341 391\ne 341 422\ne 342 343\ne 342 357\ne 342 364\ne 342 365\ne 343 363\ne 343 364\ne 343 422\ne 344 357\ne 345 355\ne 345 356\ne 345 393\ne 346 380\ne 346 382\ne 346 389\ne 346 394\ne 347 360\ne 347 364\ne 347 373\ne 347 393\ne 347 394\ne 347 395\ne 348 356\ne 348 382\ne 348 395\ne 349 361\ne 349 365\ne 349 374\ne 349 380\ne 349 396\ne 349 397\ne 350 398\ne 351 398\ne 351 399\ne 352 387\ne 355 367\ne 355 370\ne 356 367\ne 356 400\ne 358 360\ne 358 379\ne 358 404\ne 358 420\ne 359 361\ne 359 379\ne 359 386\ne 360 373\ne 360 379\ne 361 374\ne 361 379\ne 361 401\ne 362 379\ne 362 386\ne 363 365\ne 363 387\ne 363 407\ne 363 422\ne 364 366\ne 364 394\ne 364 402\ne 365 380\ne 365 403\ne 366 381\ne 366 385\ne 366 393\ne 366 404\ne 366 422\ne 366 423\ne 367 405\ne 367 407\ne 367 412\ne 367 415\ne 367 422\ne 367 427\ne 367 564\ne 368 391\ne 368 398\ne 368 406\ne 369 385\ne 369 392\ne 369 396\ne 369 399\ne 369 810\ne 370 398\ne 370 403\ne 370 405\ne 371 390\ne 371 397\ne 371 408\ne 371 418\ne 372 393\ne 372 400\ne 372 423\ne 372 424\ne 373 375\ne 373 409\ne 374 376\ne 374 410\ne 375 378\ne 375 388\ne 375 393\ne 375 411\ne 375 425\ne 376 412\ne 377 406\ne 378 389\ne 379 413\ne 380 382\ne 380 414\ne 381 383\ne 381 385\ne 381 393\ne 381 399\ne 382 415\ne 383 391\ne 383 394\ne 383 399\ne 383 402\ne 383 406\ne 383 416\ne 383 418\ne 384 394\ne 384 418\ne 387 407\ne 388 389\ne 388 393\ne 388 420\ne 389 390\ne 389 421\ne 392 399\ne 392 402\ne 392 404\ne 393 400\ne 395 400\ne 397 841\ne 401 410\ne 401 413\ne 402 404\ne 402 417\ne 403 405\ne 403 414\ne 403 417\ne 404 417\ne 405 415\ne 405 417\ne 406 417\ne 406 418\ne 408 425\ne 408 426\ne 409 411\ne 410 412\ne 411 420\ne 414 415\ne 415 422\ne 416 426\ne 419 426\ne 420 421\ne 421 426\ne 422 423\ne 423 424\ne 423 427\ne 424 427\ne 428 429\ne 428 430\ne 428 431\ne 428 432\ne 428 433\ne 428 435\ne 428 442\ne 428 444\ne 428 451\ne 428 459\ne 428 467\ne 428 469\ne 428 476\ne 428 479\ne 428 485\ne 428 487\ne 428 496\ne 428 504\ne 428 513\ne 428 514\ne 428 520\ne 428 522\ne 428 529\ne 428 541\ne 428 547\ne 428 552\ne 428 556\ne 428 562\ne 428 564\ne 429 431\ne 429 433\ne 429 437\ne 429 445\ne 429 536\ne 429 537\ne 429 538\ne 429 539\ne 429 540\ne 429 652\ne 429 657\ne 429 661\ne 429 663\ne 429 667\ne 429 707\ne 429 712\ne 429 720\ne 429 726\ne 429 729\ne 429 736\ne 429 740\ne 429 741\ne 429 745\ne 429 746\ne 430 431\ne 430 432\ne 430 466\ne 430 474\ne 430 483\ne 430 492\ne 430 508\ne 430 515\ne 430 524\ne 430 530\ne 430 787\ne 430 790\ne 430 793\ne 430 801\ne 430 806\ne 430 810\ne 430 814\ne 430 822\ne 430 826\ne 430 836\ne 430 838\ne 430 842\ne 431 440\ne 431 462\ne 431 471\ne 431 488\ne 431 560\ne 431 561\ne 431 734\ne 431 737\ne 431 740\ne 431 756\ne 431 769\ne 431 772\ne 431 779\ne 431 783\ne 431 788\ne 431 789\ne 431 793\ne 431 794\ne 431 795\ne 432 512\ne 432 518\ne 432 526\ne 432 534\ne 432 545\ne 432 549\ne 432 553\ne 432 557\ne 432 580\ne 432 587\ne 432 590\ne 432 595\ne 432 600\ne 432 807\ne 432 808\ne 432 809\ne 432 810\ne 432 849\ne 432 852\ne 432 859\ne 432 864\ne 432 874\ne 433 497\ne 433 498\ne 433 499\ne 433 500\ne 433 501\ne 433 502\ne 433 503\ne 433 569\ne 433 576\ne 433 627\ne 433 642\ne 433 647\ne 433 649\ne 433 656\ne 433 660\ne 433 662\ne 433 667\ne 433 675\ne 433 676\ne 433 677\ne 433 679\ne 433 682\ne 434 438\ne 434 459\ne 434 485\ne 434 510\ne 434 572\ne 434 617\ne 434 666\ne 434 696\ne 434 754\ne 434 797\ne 435 478\ne 435 479\ne 435 480\ne 435 481\ne 435 482\ne 435 483\ne 435 484\ne 435 485\ne 435 567\ne 435 595\ne 435 607\ne 435 643\ne 435 661\ne 436 439\ne 436 485\ne 436 487\ne 436 499\ne 436 533\ne 436 576\ne 436 621\ne 436 668\ne 437 440\ne 437 458\ne 437 485\ne 437 523\ne 437 537\ne 437 593\ne 437 661\ne 437 668\ne 437 669\ne 437 699\ne 437 700\ne 437 701\ne 438 441\ne 438 478\ne 438 485\ne 438 504\ne 438 543\ne 438 584\ne 438 585\ne 438 640\ne 438 666\ne 438 699\ne 438 755\ne 438 799\ne 439 480\ne 439 485\ne 439 529\ne 439 605\ne 439 617\ne 439 619\ne 439 700\ne 440 485\ne 440 499\ne 440 503\ne 440 523\ne 440 526\ne 440 754\ne 440 755\ne 440 756\ne 440 809\ne 440 822\ne 440 870\ne 441 481\ne 441 485\ne 441 541\ne 441 636\ne 441 641\ne 441 701\ne 442 468\ne 442 485\ne 442 495\ne 442 510\ne 442 533\ne 442 537\ne 442 543\ne 442 552\ne 442 756\ne 442 801\ne 442 843\ne 443 447\ne 443 469\ne 443 496\ne 443 517\ne 443 623\ne 443 671\ne 443 698\ne 443 702\ne 443 757\ne 443 802\ne 444 486\ne 444 487\ne 444 488\ne 444 489\ne 444 490\ne 444 491\ne 444 492\ne 444 493\ne 444 494\ne 444 495\ne 444 496\ne 444 568\ne 444 600\ne 444 611\ne 444 621\ne 444 642\ne 444 663\ne 445 465\ne 445 483\ne 445 496\ne 445 507\ne 445 524\ne 445 538\ne 445 569\ne 445 643\ne 445 663\ne 445 702\ne 445 703\ne 445 704\ne 445 705\ne 445 706\ne 446 448\ne 446 466\ne 446 496\ne 446 504\ne 446 530\ne 446 544\ne 446 570\ne 446 583\ne 446 585\ne 446 644\ne 446 704\ne 446 739\ne 446 803\ne 447 449\ne 447 486\ne 447 496\ne 447 514\ne 447 548\ne 447 645\ne 447 671\ne 447 672\ne 447 705\ne 447 739\ne 447 763\ne 447 804\ne 447 846\ne 448 489\ne 448 496\ne 448 507\ne 448 512\ne 448 530\ne 448 535\ne 448 541\ne 448 572\ne 448 623\ne 448 625\ne 448 634\ne 448 754\ne 448 758\ne 448 763\ne 448 771\ne 448 805\ne 448 809\ne 448 814\ne 448 834\ne 448 847\ne 448 866\ne 449 490\ne 449 496\ne 449 547\ne 449 573\ne 449 646\ne 449 706\ne 449 848\ne 450 491\ne 450 496\ne 450 524\ne 450 526\ne 450 552\ne 450 574\ne 450 768\ne 450 810\ne 451 477\ne 451 496\ne 451 503\ne 451 511\ne 451 517\ne 451 538\ne 451 544\ne 451 548\ne 451 556\ne 451 575\ne 451 806\ne 451 849\ne 452 456\ne 452 474\ne 452 493\ne 452 504\ne 452 515\ne 452 531\ne 452 545\ne 452 570\ne 452 577\ne 452 581\ne 452 592\ne 452 628\ne 452 629\ne 452 714\ne 453 457\ne 453 475\ne 453 514\ne 453 536\ne 453 549\ne 453 582\ne 453 589\ne 453 596\ne 453 630\ne 453 631\ne 453 648\ne 453 657\ne 453 676\ne 453 714\ne 453 735\ne 454 483\ne 454 495\ne 454 522\ne 454 530\ne 454 537\ne 454 553\ne 454 571\ne 454 578\ne 454 583\ne 454 593\ne 454 632\ne 454 633\ne 454 634\ne 454 663\ne 454 679\ne 454 724\ne 455 458\ne 455 529\ne 455 557\ne 455 584\ne 455 597\ne 455 635\ne 455 636\ne 455 664\ne 455 668\ne 455 682\ne 456 497\ne 456 541\ne 456 572\ne 456 598\ne 456 604\ne 456 617\ne 456 637\ne 456 873\ne 457 498\ne 457 515\ne 457 519\ne 457 536\ne 457 540\ne 457 547\ne 457 573\ne 457 579\ne 457 594\ne 457 599\ne 457 618\ne 457 623\ne 457 638\ne 457 807\ne 457 814\ne 457 841\ne 458 500\ne 458 531\ne 458 535\ne 458 537\ne 458 556\ne 458 575\ne 458 586\ne 458 605\ne 458 639\ne 458 809\ne 459 504\ne 459 505\ne 459 506\ne 459 507\ne 459 508\ne 459 509\ne 459 510\ne 459 511\ne 459 512\ne 459 513\ne 459 581\ne 459 649\ne 459 707\ne 459 722\ne 460 463\ne 460 502\ne 460 513\ne 460 514\ne 460 550\ne 460 582\ne 460 667\ne 460 728\ne 460 750\ne 461 465\ne 461 493\ne 461 503\ne 461 513\ne 461 529\ne 461 536\ne 461 544\ne 461 558\ne 461 584\ne 461 596\ne 461 599\ne 461 619\ne 461 632\ne 461 644\ne 461 651\ne 461 663\ne 461 674\ne 461 708\ne 461 722\ne 461 738\ne 461 743\ne 461 755\ne 461 765\ne 461 776\ne 462 466\ne 462 513\ne 462 560\ne 462 652\ne 462 670\ne 462 672\ne 462 704\ne 462 743\ne 462 759\ne 462 760\ne 463 505\ne 463 513\ne 463 547\ne 463 653\ne 463 750\ne 463 757\ne 463 776\ne 464 468\ne 464 506\ne 464 513\ne 464 552\ne 464 565\ne 464 585\ne 464 654\ne 464 709\ne 464 755\ne 464 759\ne 464 811\ne 464 825\ne 464 850\ne 464 870\ne 465 507\ne 465 513\ne 465 536\ne 465 556\ne 465 573\ne 465 586\ne 465 655\ne 465 710\ne 465 812\ne 465 821\ne 466 508\ne 466 513\ne 466 540\ne 466 544\ne 466 587\ne 466 609\ne 466 811\ne 466 812\ne 466 813\ne 466 814\ne 466 851\ne 467 513\ne 467 519\ne 467 527\ne 467 535\ne 467 539\ne 467 550\ne 467 554\ne 467 558\ne 467 560\ne 467 562\ne 467 565\ne 467 588\ne 467 606\ne 467 611\ne 467 814\ne 467 852\ne 468 510\ne 468 513\ne 468 571\ne 468 632\ne 468 711\ne 468 760\ne 468 821\ne 468 825\ne 468 855\ne 469 514\ne 469 515\ne 469 516\ne 469 517\ne 469 518\ne 469 519\ne 469 520\ne 469 589\ne 469 656\ne 469 712\ne 469 727\ne 469 734\ne 470 473\ne 470 520\ne 470 529\ne 470 559\ne 470 596\ne 470 598\ne 470 713\ne 470 815\ne 470 853\ne 471 474\ne 471 502\ne 471 512\ne 471 520\ne 471 542\ne 471 549\ne 471 561\ne 471 657\ne 471 672\ne 471 690\ne 471 734\ne 471 761\ne 471 762\ne 471 763\ne 471 764\ne 471 765\ne 472 475\ne 472 520\ne 472 541\ne 472 563\ne 472 625\ne 472 629\ne 472 635\ne 472 714\ne 472 761\ne 472 816\ne 473 477\ne 473 520\ne 473 556\ne 473 566\ne 473 572\ne 473 683\ne 473 684\ne 473 716\ne 473 763\ne 473 854\ne 474 515\ne 474 520\ne 474 542\ne 474 590\ne 474 609\ne 474 638\ne 474 815\ne 474 816\ne 475 516\ne 475 520\ne 475 549\ne 475 562\ne 475 580\ne 475 604\ne 475 606\ne 475 615\ne 476 520\ne 476 540\ne 476 546\ne 476 559\ne 476 561\ne 476 563\ne 476 564\ne 476 566\ne 476 591\ne 476 612\ne 476 627\ne 476 851\ne 477 517\ne 477 520\ne 477 584\ne 477 635\ne 477 659\ne 477 717\ne 477 765\ne 477 860\ne 478 481\ne 478 504\ne 478 583\ne 478 584\ne 478 592\ne 478 691\ne 478 711\ne 478 817\ne 478 855\ne 479 521\ne 479 522\ne 479 523\ne 479 524\ne 479 525\ne 479 526\ne 479 527\ne 479 593\ne 479 643\ne 479 660\ne 479 721\ne 480 482\ne 480 529\ne 480 537\ne 480 605\ne 480 667\ne 481 484\ne 481 521\ne 481 541\ne 481 634\ne 481 636\ne 481 671\ne 481 673\ne 481 718\ne 481 819\ne 481 857\ne 482 523\ne 482 556\ne 482 689\ne 482 691\ne 482 693\ne 483 524\ne 483 537\ne 483 571\ne 483 575\ne 483 595\ne 483 817\ne 483 818\ne 483 819\ne 483 820\ne 483 821\ne 483 822\ne 484 525\ne 484 562\ne 484 655\ne 484 719\ne 484 821\ne 485 607\ne 485 621\ne 485 822\ne 485 859\ne 486 490\ne 486 514\ne 486 596\ne 486 672\ne 486 694\ne 486 717\ne 486 743\ne 486 766\ne 486 823\ne 486 860\ne 487 528\ne 487 529\ne 487 530\ne 487 531\ne 487 532\ne 487 533\ne 487 534\ne 487 535\ne 487 597\ne 487 644\ne 487 662\ne 487 720\ne 487 737\ne 488 492\ne 488 511\ne 488 526\ne 488 544\ne 488 553\ne 488 663\ne 488 721\ne 488 737\ne 488 766\ne 488 767\ne 488 768\ne 489 493\ne 489 512\ne 489 541\ne 489 557\ne 489 598\ne 489 664\ne 489 673\ne 489 722\ne 489 767\ne 489 792\ne 489 824\ne 489 861\ne 490 494\ne 490 528\ne 490 547\ne 490 599\ne 490 665\ne 490 723\ne 490 743\ne 490 744\ne 490 792\ne 490 827\ne 490 862\ne 491 495\ne 491 552\ne 491 666\ne 491 724\ne 491 797\ne 491 799\ne 492 530\ne 492 571\ne 492 574\ne 492 600\ne 492 768\ne 492 823\ne 492 824\ne 492 825\ne 492 826\ne 492 866\ne 493 531\ne 493 544\ne 493 557\ne 493 562\ne 493 570\ne 493 575\ne 493 584\ne 493 587\ne 493 601\ne 493 611\ne 493 654\ne 493 694\ne 493 695\ne 493 799\ne 493 817\ne 493 825\ne 493 863\ne 493 873\ne 494 532\ne 494 564\ne 494 602\ne 494 725\ne 494 827\ne 495 533\ne 495 553\ne 495 578\ne 495 580\ne 495 621\ne 496 569\ne 496 603\ne 496 826\ne 496 864\ne 497 501\ne 497 518\ne 497 535\ne 497 541\ne 497 549\ne 497 558\ne 497 604\ne 497 641\ne 497 664\ne 497 678\ne 497 727\ne 497 741\ne 498 502\ne 498 519\ne 498 547\ne 498 560\ne 498 646\ne 498 653\ne 498 665\ne 498 670\ne 498 676\ne 498 681\ne 498 728\ne 498 734\ne 498 745\ne 499 526\ne 499 552\ne 499 557\ne 499 654\ne 499 658\ne 499 666\ne 499 668\ne 499 679\ne 499 696\ne 499 697\ne 499 737\ne 499 746\ne 499 782\ne 500 503\ne 500 556\ne 500 605\ne 500 655\ne 500 671\ne 500 682\ne 500 698\ne 500 738\ne 501 536\ne 501 562\ne 501 579\ne 501 584\ne 501 586\ne 501 637\ne 501 683\ne 501 688\ne 501 691\ne 502 549\ne 502 560\ne 502 564\ne 502 582\ne 502 587\ne 502 606\ne 502 627\ne 502 638\ne 502 672\ne 502 680\ne 502 684\ne 502 692\ne 502 694\ne 503 538\ne 503 558\ne 503 569\ne 503 575\ne 503 607\ne 503 659\ne 503 669\ne 503 674\ne 503 689\ne 504 541\ne 504 542\ne 504 543\ne 504 544\ne 504 545\ne 504 546\ne 504 729\ne 504 769\ne 504 774\ne 505 509\ne 505 540\ne 505 547\ne 505 561\ne 505 670\ne 505 740\ne 505 774\ne 506 510\ne 506 552\ne 506 824\ne 506 830\ne 506 865\ne 507 511\ne 507 535\ne 507 556\ne 507 560\ne 507 671\ne 507 681\ne 507 684\ne 507 693\ne 507 696\ne 507 704\ne 507 722\ne 507 730\ne 507 737\ne 507 741\ne 507 758\ne 507 770\ne 507 791\ne 507 802\ne 507 804\ne 507 819\ne 507 830\ne 508 512\ne 508 744\ne 508 758\ne 508 767\ne 508 797\ne 508 827\ne 508 828\ne 509 542\ne 509 564\ne 509 609\ne 509 672\ne 509 731\ne 509 804\ne 509 823\ne 510 543\ne 510 571\ne 510 572\ne 510 610\ne 510 673\ne 510 732\ne 510 771\ne 510 819\ne 510 824\ne 511 544\ne 511 560\ne 511 611\ne 511 665\ne 511 674\ne 511 733\ne 511 827\ne 511 866\ne 512 545\ne 512 561\ne 512 581\ne 512 587\ne 512 612\ne 512 690\ne 512 828\ne 512 865\ne 512 866\ne 513 587\ne 513 613\ne 513 644\ne 513 652\ne 513 675\ne 514 547\ne 514 548\ne 514 549\ne 514 550\ne 514 676\ne 514 727\ne 514 772\ne 514 787\ne 515 518\ne 515 540\ne 515 563\ne 515 714\ne 515 734\ne 515 741\ne 515 744\ne 515 774\ne 515 787\ne 515 798\ne 515 802\ne 515 829\ne 516 519\ne 516 562\ne 516 604\ne 516 695\ne 516 698\ne 516 735\ne 516 868\ne 517 548\ne 517 584\ne 517 614\ne 517 665\ne 517 748\ne 517 749\ne 517 776\ne 518 549\ne 518 563\ne 518 577\ne 518 579\ne 518 589\ne 518 590\ne 518 615\ne 518 627\ne 518 690\ne 518 867\ne 518 868\ne 519 550\ne 519 576\ne 519 579\ne 519 606\ne 519 648\ne 519 688\ne 520 590\ne 520 606\ne 520 612\ne 520 616\ne 520 657\ne 520 677\ne 521 525\ne 521 541\ne 521 617\ne 521 671\ne 521 678\ne 521 751\ne 521 754\ne 521 830\ne 521 869\ne 522 551\ne 522 552\ne 522 553\ne 522 554\ne 522 679\ne 522 721\ne 522 780\ne 523 556\ne 523 689\ne 523 740\ne 524 526\ne 524 724\ne 524 746\ne 524 830\ne 524 831\ne 524 832\ne 525 527\ne 525 551\ne 525 562\ne 525 619\ne 525 654\ne 525 655\ne 525 699\ne 525 743\ne 525 777\ne 526 553\ne 526 569\ne 526 574\ne 526 593\ne 526 595\ne 526 621\ne 526 666\ne 526 869\ne 526 870\ne 527 554\ne 527 622\ne 527 733\ne 527 778\ne 527 832\ne 528 532\ne 528 547\ne 528 623\ne 528 681\ne 528 744\ne 528 753\ne 528 757\ne 528 758\ne 528 833\ne 528 871\ne 529 555\ne 529 556\ne 529 557\ne 529 558\ne 529 559\ne 529 682\ne 529 722\ne 529 736\ne 529 779\ne 529 790\ne 530 534\ne 530 724\ne 530 737\ne 530 780\ne 530 790\ne 530 833\ne 530 834\ne 530 835\ne 531 535\ne 531 562\ne 531 624\ne 531 683\ne 531 699\ne 531 738\ne 531 742\ne 531 834\ne 532 555\ne 532 564\ne 532 625\ne 532 684\ne 532 739\ne 532 758\ne 532 761\ne 532 781\ne 532 835\ne 532 846\ne 533 685\ne 533 782\ne 533 855\ne 533 857\ne 534 557\ne 534 570\ne 534 572\ne 534 578\ne 534 597\ne 534 600\ne 534 666\ne 534 871\ne 535 558\ne 535 569\ne 535 572\ne 535 581\ne 535 586\ne 535 611\ne 535 626\ne 535 644\ne 535 664\ne 535 671\ne 535 686\ne 535 716\ne 535 732\ne 535 753\ne 535 846\ne 535 857\ne 535 869\ne 536 539\ne 536 562\ne 536 688\ne 536 719\ne 536 735\ne 536 738\ne 536 741\ne 536 794\ne 536 841\ne 537 711\ne 537 732\ne 537 742\ne 537 746\ne 537 756\ne 537 790\ne 538 689\ne 538 717\ne 538 733\ne 538 743\ne 538 791\ne 539 560\ne 539 582\ne 539 652\ne 539 671\ne 539 674\ne 539 748\ne 539 750\ne 539 751\ne 540 561\ne 540 581\ne 540 609\ne 540 627\ne 540 657\ne 540 670\ne 540 744\ne 540 749\ne 540 752\ne 540 753\ne 541 562\ne 541 563\ne 541 741\ne 541 783\ne 541 836\ne 541 840\ne 542 546\ne 542 564\ne 542 774\ne 542 793\ne 542 828\ne 542 840\ne 543 628\ne 543 742\ne 543 784\ne 544 709\ne 544 711\ne 544 743\ne 544 747\ne 544 749\ne 544 767\ne 544 785\ne 544 790\ne 544 794\ne 544 812\ne 544 825\ne 544 837\ne 544 860\ne 544 862\ne 545 629\ne 545 761\ne 545 825\ne 545 834\ne 545 855\ne 546 563\ne 546 690\ne 546 744\ne 546 786\ne 546 828\ne 546 862\ne 546 871\ne 547 564\ne 547 745\ne 547 774\ne 547 807\ne 547 838\ne 547 844\ne 548 630\ne 548 747\ne 548 748\ne 548 839\ne 549 761\ne 549 787\ne 549 794\ne 549 807\ne 549 840\ne 549 845\ne 549 846\ne 549 856\ne 549 860\ne 550 631\ne 550 688\ne 550 716\ne 550 717\ne 550 841\ne 551 554\ne 551 562\ne 551 632\ne 551 691\ne 551 742\ne 551 743\ne 551 760\ne 551 817\ne 552 565\ne 552 746\ne 552 780\ne 552 788\ne 552 808\ne 552 843\ne 553 756\ne 553 782\ne 553 808\ne 554 565\ne 554 634\ne 554 693\ne 554 732\ne 554 733\ne 554 758\ne 555 559\ne 555 564\ne 555 635\ne 555 694\ne 555 747\ne 555 761\ne 555 765\ne 555 823\ne 555 825\ne 555 872\ne 556 566\ne 556 789\ne 556 809\ne 556 842\ne 557 782\ne 557 790\ne 557 809\ne 557 843\ne 557 853\ne 557 872\ne 557 873\ne 558 636\ne 558 748\ne 558 791\ne 558 873\ne 559 566\ne 559 598\ne 559 599\ne 559 695\ne 559 749\ne 559 792\ne 559 825\ne 560 750\ne 560 778\ne 560 791\ne 560 794\ne 560 796\ne 560 814\ne 560 844\ne 561 786\ne 561 792\ne 561 795\ne 561 807\ne 561 813\ne 562 637\ne 562 794\ne 563 629\ne 563 810\ne 563 840\ne 564 638\ne 564 795\ne 564 840\ne 564 844\ne 564 851\ne 564 874\ne 565 696\ne 565 751\ne 565 758\ne 565 796\ne 565 797\ne 565 869\ne 565 870\ne 566 572\ne 566 573\ne 566 639\ne 566 698\ne 566 753\ne 566 802\ne 566 871\ne 567 592\ne 567 593\ne 567 594\ne 567 595\ne 568 596\ne 568 597\ne 568 598\ne 568 599\ne 568 600\ne 568 601\ne 568 602\ne 568 603\ne 568 642\ne 569 586\ne 569 595\ne 569 603\ne 569 642\ne 569 643\ne 569 644\ne 569 645\ne 569 646\ne 570 572\ne 570 587\ne 570 603\ne 570 644\ne 570 684\ne 570 687\ne 570 802\ne 571 574\ne 571 603\ne 572 598\ne 572 603\ne 572 612\ne 572 626\ne 572 847\ne 572 852\ne 572 873\ne 573 599\ne 573 603\ne 573 646\ne 573 802\ne 573 848\ne 574 603\ne 574 621\ne 574 822\ne 575 603\ne 575 607\ne 575 611\ne 575 614\ne 575 630\ne 575 822\ne 575 849\ne 575 873\ne 576 604\ne 576 605\ne 576 606\ne 576 607\ne 576 608\ne 576 647\ne 576 648\ne 577 590\ne 577 601\ne 577 608\ne 577 624\ne 577 629\ne 578 595\ne 578 608\ne 578 642\ne 578 666\ne 578 685\ne 579 608\ne 579 627\ne 579 852\ne 579 868\ne 580 608\ne 580 612\ne 580 615\ne 580 621\ne 580 629\ne 581 609\ne 581 610\ne 581 611\ne 581 612\ne 581 613\ne 581 649\ne 581 657\ne 581 664\ne 582 606\ne 582 613\ne 582 631\ne 582 647\ne 582 657\ne 583 585\ne 583 613\ne 583 634\ne 583 650\ne 583 711\ne 584 586\ne 584 601\ne 584 607\ne 584 613\ne 584 636\ne 584 642\ne 584 651\ne 584 664\ne 584 683\ne 585 613\ne 585 654\ne 585 850\ne 585 863\ne 586 613\ne 586 655\ne 586 711\ne 587 613\ne 587 627\ne 587 850\ne 587 851\ne 587 852\ne 588 613\ne 588 622\ne 588 626\ne 588 631\ne 588 634\ne 588 636\ne 588 637\ne 588 852\ne 589 614\ne 589 615\ne 589 616\ne 589 656\ne 590 616\ne 590 853\ne 590 854\ne 591 616\ne 591 627\ne 591 638\ne 591 639\ne 592 685\ne 592 855\ne 593 617\ne 593 618\ne 593 619\ne 593 620\ne 593 621\ne 593 622\ne 593 660\ne 593 663\ne 593 668\ne 594 618\ne 594 719\ne 594 858\ne 595 855\ne 595 856\ne 595 857\ne 595 858\ne 595 859\ne 596 599\ne 596 659\ne 596 860\ne 597 623\ne 597 624\ne 597 625\ne 597 626\ne 597 662\ne 598 601\ne 598 612\ne 598 664\ne 598 861\ne 599 602\ne 599 623\ne 599 665\ne 599 690\ne 599 765\ne 599 862\ne 599 866\ne 600 860\ne 600 861\ne 600 862\ne 600 863\ne 600 864\ne 601 624\ne 601 637\ne 601 855\ne 601 863\ne 602 625\ne 602 638\ne 602 866\ne 603 864\ne 604 615\ne 604 626\ne 604 636\ne 605 607\ne 605 683\ne 605 686\ne 606 638\ne 607 636\ne 607 669\ne 609 638\ne 609 672\ne 609 846\ne 609 860\ne 610 628\ne 610 673\ne 610 857\ne 610 861\ne 611 674\ne 611 717\ne 611 797\ne 611 866\ne 612 629\ne 613 675\ne 614 630\ne 614 695\ne 614 798\ne 616 677\ne 617 619\ne 617 678\ne 617 686\ne 617 696\ne 617 701\ne 617 722\ne 617 869\ne 618 620\ne 618 723\ne 618 832\ne 619 622\ne 619 632\ne 619 637\ne 619 640\ne 620 633\ne 620 638\ne 620 680\ne 620 725\ne 620 832\ne 622 634\ne 622 674\ne 623 625\ne 623 681\ne 623 690\ne 623 698\ne 623 738\ne 623 763\ne 623 867\ne 623 871\ne 624 626\ne 624 637\ne 624 640\ne 624 683\ne 624 854\ne 624 867\ne 625 635\ne 625 638\ne 625 684\ne 625 854\ne 626 636\ne 626 673\ne 626 686\ne 626 698\ne 627 677\ne 627 687\ne 627 690\ne 627 695\ne 627 697\ne 627 698\ne 628 829\ne 629 863\ne 630 694\ne 630 717\ne 630 800\ne 630 818\ne 631 659\ne 632 634\ne 632 637\ne 632 691\ne 632 755\ne 632 821\ne 632 834\ne 632 855\ne 633 638\ne 633 692\ne 633 725\ne 633 835\ne 634 673\ne 634 674\ne 634 693\ne 634 754\ne 635 638\ne 635 694\ne 635 815\ne 635 860\ne 635 863\ne 635 872\ne 636 873\ne 638 814\ne 638 874\ne 639 698\ne 639 868\ne 640 641\ne 640 699\ne 641 701\ne 641 845\ne 642 662\ne 642 663\ne 642 664\ne 642 665\ne 642 666\ne 643 655\ne 643 667\ne 643 671\ne 643 689\ne 643 703\ne 644 704\ne 644 757\ne 644 855\ne 645 646\ne 645 676\ne 645 705\ne 645 710\ne 645 716\ne 645 856\ne 646 665\ne 646 706\ne 646 710\ne 646 753\ne 646 757\ne 646 858\ne 647 667\ne 647 668\ne 647 669\ne 648 676\ne 648 688\ne 649 670\ne 649 671\ne 649 672\ne 649 673\ne 649 674\ne 649 675\ne 649 707\ne 650 654\ne 650 675\ne 650 679\ne 650 693\ne 650 704\ne 651 655\ne 651 669\ne 651 675\ne 651 682\ne 651 688\ne 651 699\ne 651 708\ne 651 717\ne 652 675\ne 652 707\ne 652 708\ne 652 709\ne 652 710\ne 652 711\ne 653 670\ne 653 675\ne 653 702\ne 654 675\ne 654 696\ne 654 699\ne 654 704\ne 654 709\ne 654 817\ne 655 671\ne 655 675\ne 655 688\ne 655 710\ne 656 676\ne 656 677\ne 656 712\ne 657 677\ne 657 712\ne 657 713\ne 657 714\ne 657 715\ne 657 716\ne 657 717\ne 658 677\ne 658 697\ne 658 715\ne 658 853\ne 659 677\ne 659 717\ne 660 678\ne 660 679\ne 660 680\ne 661 669\ne 661 718\ne 661 719\ne 662 681\ne 662 682\ne 662 683\ne 662 684\ne 662 685\ne 662 686\ne 662 687\ne 662 720\ne 663 674\ne 663 720\ne 663 721\ne 663 722\ne 663 723\ne 663 724\ne 663 725\ne 664 713\ne 664 722\ne 664 749\ne 664 860\ne 665 681\ne 665 708\ne 665 723\ne 665 749\ne 666 724\ne 667 688\ne 667 689\ne 667 726\ne 667 727\ne 667 728\ne 668 700\ne 668 720\ne 669 689\ne 670 672\ne 670 726\ne 670 734\ne 671 674\ne 671 686\ne 671 720\ne 671 727\ne 671 730\ne 671 738\ne 671 748\ne 672 731\ne 673 732\ne 674 733\ne 674 754\ne 678 700\ne 679 691\ne 679 692\ne 679 693\ne 679 737\ne 680 692\ne 680 778\ne 681 684\ne 681 702\ne 682 694\ne 682 695\ne 682 736\ne 683 686\ne 683 738\ne 684 687\ne 684 694\ne 684 714\ne 684 739\ne 684 802\ne 687 695\ne 688 727\ne 689 708\ne 689 748\ne 690 744\ne 691 693\ne 691 708\ne 691 711\ne 692 781\ne 693 696\ne 693 718\ne 694 695\ne 694 714\ne 694 717\ne 694 747\ne 694 791\ne 695 698\ne 695 749\ne 696 701\ne 696 751\ne 696 819\ne 696 873\ne 697 752\ne 697 850\ne 698 753\ne 698 867\ne 699 701\ne 699 729\ne 699 742\ne 699 755\ne 700 736\ne 701 718\ne 701 741\ne 701 798\ne 701 800\ne 702 705\ne 702 712\ne 702 757\ne 703 710\ne 703 726\ne 703 730\ne 704 729\ne 704 737\ne 704 743\ne 704 817\ne 705 706\ne 705 818\ne 706 723\ne 706 745\ne 706 820\ne 707 729\ne 707 730\ne 707 731\ne 707 732\ne 707 733\ne 708 710\ne 708 736\ne 708 743\ne 708 748\ne 709 711\ne 709 746\ne 709 751\ne 709 759\ne 709 777\ne 710 730\ne 710 777\ne 711 732\ne 711 760\ne 712 734\ne 712 735\ne 713 716\ne 713 736\ne 713 749\ne 714 741\ne 714 761\ne 715 746\ne 715 752\ne 715 762\ne 715 815\ne 716 717\ne 716 753\ne 716 763\ne 717 765\ne 718 719\ne 718 741\ne 718 818\ne 720 736\ne 720 737\ne 720 738\ne 720 739\ne 721 733\ne 721 740\ne 721 743\ne 722 741\ne 722 767\ne 722 823\ne 723 725\ne 723 745\ne 723 776\ne 724 746\ne 724 754\ne 724 755\ne 725 739\ne 725 765\ne 725 823\ne 726 740\ne 727 741\ne 727 748\ne 728 745\ne 728 750\ne 729 741\ne 729 742\ne 729 743\ne 729 744\ne 729 769\ne 730 733\ne 730 750\ne 730 757\ne 730 770\ne 731 766\ne 732 742\ne 732 767\ne 732 771\ne 732 869\ne 733 743\ne 733 750\ne 734 757\ne 734 772\ne 734 773\ne 734 774\ne 734 775\ne 734 776\ne 736 747\ne 736 748\ne 736 749\ne 736 779\ne 737 779\ne 737 780\ne 737 781\ne 737 782\ne 738 763\ne 738 773\ne 739 747\ne 739 763\ne 739 770\ne 739 781\ne 740 750\ne 741 783\ne 741 787\ne 741 791\ne 742 784\ne 742 817\ne 743 779\ne 743 785\ne 743 791\ne 744 786\ne 746 751\ne 746 752\ne 746 788\ne 746 790\ne 747 749\ne 747 766\ne 748 791\ne 749 753\ne 749 792\ne 749 860\ne 751 754\ne 751 770\ne 751 796\ne 751 832\ne 752 811\ne 753 757\ne 753 812\ne 753 846\ne 754 755\ne 754 771\ne 754 797\ne 754 832\ne 755 769\ne 755 784\ne 755 799\ne 755 834\ne 756 760\ne 756 764\ne 756 771\ne 756 775\ne 756 782\ne 756 784\ne 756 788\ne 756 801\ne 756 809\ne 757 776\ne 757 802\ne 758 767\ne 758 770\ne 758 780\ne 758 783\ne 758 805\ne 758 812\ne 758 827\ne 758 842\ne 758 844\ne 759 760\ne 759 788\ne 759 796\ne 759 811\ne 759 831\ne 759 837\ne 760 771\ne 760 817\ne 760 837\ne 761 783\ne 761 816\ne 762 764\ne 762 788\ne 762 865\ne 763 765\ne 763 773\ne 763 789\ne 763 841\ne 764 775\ne 764 854\ne 765 776\ne 765 823\ne 765 841\ne 765 866\ne 766 772\ne 766 823\ne 767 783\ne 767 790\ne 767 824\ne 767 869\ne 768 780\ne 768 793\ne 768 827\ne 768 837\ne 769 783\ne 769 784\ne 769 785\ne 769 786\ne 770 789\ne 771 784\ne 772 787\ne 773 776\ne 773 789\ne 774 793\ne 775 829\ne 775 867\ne 777 778\ne 777 794\ne 778 832\ne 779 789\ne 779 790\ne 779 791\ne 779 792\ne 780 793\ne 781 795\ne 781 803\ne 781 804\ne 781 835\ne 782 817\ne 782 819\ne 783 794\ne 783 836\ne 784 834\ne 785 823\ne 785 837\ne 786 833\ne 787 800\ne 787 804\ne 787 818\ne 787 823\ne 787 838\ne 787 839\ne 787 840\ne 787 841\ne 788 796\ne 789 842\ne 790 815\ne 790 842\ne 790 843\ne 791 839\ne 792 837\ne 793 844\ne 794 807\ne 794 812\ne 795 813\ne 796 830\ne 796 831\ne 797 799\ne 797 822\ne 797 847\ne 797 869\ne 798 800\ne 798 822\ne 798 829\ne 799 817\ne 799 822\ne 799 850\ne 800 818\ne 800 822\ne 800 845\ne 801 822\ne 801 829\ne 802 804\ne 802 826\ne 802 871\ne 803 805\ne 803 813\ne 803 826\ne 803 837\ne 803 850\ne 804 823\ne 804 826\ne 804 839\ne 804 846\ne 805 824\ne 805 826\ne 805 828\ne 805 836\ne 805 847\ne 805 871\ne 806 826\ne 806 827\ne 806 831\ne 806 837\ne 806 839\ne 806 842\ne 806 849\ne 807 838\ne 807 848\ne 807 858\ne 807 862\ne 807 871\ne 807 874\ne 808 850\ne 808 865\ne 809 834\ne 809 842\ne 809 849\ne 809 854\ne 809 867\ne 810 828\ne 810 840\ne 810 843\ne 810 851\ne 811 850\ne 812 842\ne 812 846\ne 812 848\ne 813 837\ne 813 851\ne 814 832\ne 814 841\ne 814 844\ne 814 852\ne 814 866\ne 815 853\ne 815 860\ne 815 861\ne 816 836\ne 816 872\ne 817 819\ne 817 855\ne 818 820\ne 818 856\ne 819 821\ne 819 830\ne 819 836\ne 819 857\ne 819 873\ne 820 838\ne 820 858\ne 822 859\ne 823 860\ne 824 825\ne 824 828\ne 824 836\ne 824 843\ne 824 861\ne 825 834\ne 825 837\ne 825 843\ne 825 849\ne 825 851\ne 825 863\ne 825 866\ne 826 864\ne 827 837\ne 827 844\ne 827 866\ne 828 851\ne 830 836\ne 830 869\ne 831 870\ne 833 835\ne 833 838\ne 833 871\ne 845 856\ne 845 859\ne 846 848\ne 846 860\ne 846 864\ne 847 861\ne 847 864\ne 848 862\ne 848 864\ne 849 864\ne 849 866\ne 849 870\ne 850 865\ne 852 873\ne 853 854\ne 854 867\ne 855 857\ne 856 858\ne 857 869\ne 860 862\ne 861 863\ne 862 871\ne 872 874\n"
open('/tmp/874.edge','w').write(EDGE874)
import sys
sys.argv=["nn_jax.py","--edges","/tmp/874.edge","--k","5","--restarts","64","--steps","20000","--lr","0.15","--entropy","0.05","--kicks","100","--kick-size","15","--x64","--seed","7"]
_ = main()
